# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.MAGASINS = Set(initialize=['MTLE', 'CV', 'MTLO'])
model.VETEMENTS = Set(initialize=['A', 'B', 'C', 'D', 'E'])
model.ARC = Set(dimen=2, initialize=[(i0,i1) for i0 in model.MAGASINS for i1 in model.VETEMENTS])

## 🔹 Parameters

In [ ]:
model.total = Param(model.MAGASINS, initialize={'MTLE': 100.0, 'CV': 100.0, 'MTLO': 100.0}, within=NonNegativeReals)
model.tot_vetement = Param(model.VETEMENTS, initialize={'A': 1000.0, 'B': 1000.0, 'C': 1000.0, 'D': 1000.0, 'E': 90.0}, within=NonNegativeReals)
model.stock = Param(model.MAGASINS, model.VETEMENTS, initialize={('MTLE', 'A'): 1.0, ('MTLE', 'B'): 1.0, ('MTLE', 'C'): -0.3333333333333333, ('MTLE', 'D'): -0.3333333333333333, ('MTLE', 'E'): -0.3333333333333333, ('CV', 'A'): 1.0, ('CV', 'B'): 1.0, ('CV', 'C'): -3.0, ('CV', 'D'): -3.0, ('CV', 'E'): -3.0, ('MTLO', 'A'): 0.25, ('MTLO', 'B'): 0.25, ('MTLO', 'C'): 0.25, ('MTLO', 'D'): -1.0, ('MTLO', 'E'): -1.0}, within=Reals)
model.gain = Param(model.MAGASINS, model.VETEMENTS, initialize={('MTLE', 'A'): 14.57, ('MTLE', 'B'): 14.45, ('MTLE', 'C'): 16.19, ('MTLE', 'D'): 15.9, ('MTLE', 'E'): 20.67, ('CV', 'A'): 15.77, ('CV', 'B'): 18.02, ('CV', 'C'): 18.65, ('CV', 'D'): 18.7, ('CV', 'E'): 17.5, ('MTLO', 'A'): 13.65, ('MTLO', 'B'): 15.26, ('MTLO', 'C'): 15.7, ('MTLO', 'D'): 15.95, ('MTLO', 'E'): 17.08}, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.MAGASINS, model.VETEMENTS, domain=NonNegativeReals)

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for m in model.MAGASINS:
    model.c_for_0.add(sum(model.X[m, v] for v in model.VETEMENTS) <= model.total[m])
model.c_for_1 = ConstraintList()
for v in model.VETEMENTS:
    model.c_for_1.add(sum(model.X[m, v] for m in model.MAGASINS) <= model.tot_vetement[v])
model.c_for_2 = ConstraintList()
for m in model.MAGASINS:
    model.c_for_2.add(sum(model.stock[m, v] * model.X[m, v] for v in model.VETEMENTS) >= 0)

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(sum(model.gain[m, v] * model.X[m, v] for v in model.VETEMENTS) for m in model.MAGASINS), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')